# ETS MARL — Multi-Agent RL for EU Emissions Trading

**Technology-Specific Energy Mix | Real CapEx | Construction Queues | PPO**

This notebook is a testing playground for the ETS MARL simulation.
It clones the repository, installs dependencies, runs tests, trains agents,
and visualizes results — all within Google Colab.

---

### Model Overview (v5.0)
- **8 agents** representing European electricity utilities (10 TWh/year each)
  - 4 archetypes × 2 replicas: coal-heavy, gas-dominant, mixed, near-green
- **5 technologies**: Coal, Gas CCGT, Onshore Wind, Offshore Wind, Solar PV
- **Real-data-grounded** CapEx (IRENA 2024), emission factors (IPCC AR5), deployment delays
- **EU ETS mechanics**: uniform-price auction, MSR, LRF (Fit for 55), 100% auctioning
- **PPO agents** with two-phase decisions (auction + secondary market)
- **Greening-only** constraint: agents can only add renewables, fossil can only decrease
- **MAC fuel-switching**: automatic coal→gas dispatch switching at marginal cost (€65/tCO₂)
- **Electricity revenue**: margin-based reward (output × electricity price)
- **Carry-forward obligations**: non-compliance shortfall added to next year's surrender
- **Soft agent cycling**: all agents learn every episode (non-active at reduced LR)

## 1. Setup — Clone Repository & Install Dependencies

In [1]:
import os, sys

def _is_project_root(path: str) -> bool:
    return (os.path.isdir(os.path.join(path, 'src')) and
            os.path.isdir(os.path.join(path, 'configs')))

if not _is_project_root('.'):
    _local_candidates = ['..', os.path.join('..', '..')]
    _colab_candidates = [
        os.path.join('Thesis-Energy-Auction', 'ets_marl ppo'),
        os.path.join('Thesis-Energy-Auction', 'ets_marl_ppo'),
        'Thesis-Energy-Auction',
    ]
    _found = False
    for _c in _local_candidates + _colab_candidates:
        if _is_project_root(_c):
            os.chdir(_c)
            _found = True
            break
    if not _found:
        os.system('git clone https://github.com/DJH961/Thesis-Energy-Auction.git 2>/dev/null || echo "Repo already cloned"')
        for _c in _colab_candidates:
            if _is_project_root(_c):
                os.chdir(_c)
                _found = True
                break
    if not _found:
        raise FileNotFoundError(
            f'Cannot find project root (needs src/ + configs/).\n'
            f'CWD: {os.getcwd()}  |  Contents: {os.listdir(".")}'
        )

sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')

Working directory: c:\Users\t-dhenke\OneDrive - Microsoft\Documents\VS Code\Master Thesis Code\Thesis-Energy-Auction\ets_marl_happo_current


In [2]:
# Install dependencies
!pip install -q gymnasium numpy torch pyyaml pandas matplotlib seaborn pytest numpy-groupies

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.10.0+cpu
CUDA available: False


## 2. Run Tests — Verify Everything Works

In [3]:
!python -m pytest tests/ -v

============================= test session starts =============================
platform win32 -- Python 3.13.12, pytest-9.0.2, pluggy-1.6.0 -- C:\Users\t-dhenke\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\t-dhenke\OneDrive - Microsoft\Documents\VS Code\Master Thesis Code\Thesis-Energy-Auction\ets_marl_happo_current
configfile: pyproject.toml
plugins: anyio-4.10.0
collecting ... collected 148 items

tests/test_anchors.py::TestAuctionAnchors::test_anchored_price_near_target PASSED [  0%]
tests/test_anchors.py::TestAuctionAnchors::test_anchored_qty_near_target PASSED [  1%]
tests/test_anchors.py::TestAuctionAnchors::test_anchored_invest_frac PASSED [  2%]
tests/test_anchors.py::TestAuctionAnchors::test_no_anchors_gives_midpoint PASSED [  2%]
tests/test_anchors.py::TestAuctionAnchors::test_anchors_within_bounds PASSED [  3%]
tests/test_anchors.py::TestSecondaryAnchors::test_anchored_price_mult 

## 3. Explore the Environment

In [ ]:
import sys
sys.path.insert(0, '.')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.environment.ets_environment import ETSEnvironment
from src.environment.company import TECH_NAMES

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

env = ETSEnvironment(config, seed=42)
obs1, _ = env.reset()

print('=== Environment Summary ===')
print(f'Agents: {env.n_agents}')
print(f'Years: {env.n_years}')
print(f'Technologies: {TECH_NAMES}')
print(f'Phase 1 obs dim: {obs1.shape[1]}')
print(f'Initial cap: {env.cap_schedule.get_cap(0):.2f} Mt')
print()

tech_names = config['technologies']['names']
for i, c in enumerate(env.companies):
    mix_str = ' | '.join([f'{tech_names[t]}: {c.mix[t]*100:.0f}%' for t in range(5)])
    print(f'A{i+1}: {mix_str}')
    print(f'     Emissions: {c.compute_emissions():.2f} Mt | '
          f'Green: {c.green_frac*100:.0f}% | '
          f'Op. Cost: {c.compute_operational_cost():.0f} M\u20ac/yr | '
          f'Avg EF: {c.weighted_emission_factor:.3f} tCO2/MWh')

## 4. Run a Random-Policy Episode (Baseline)

In [ ]:
env = ETSEnvironment(config, seed=42)
obs1, _ = env.reset(seed=42)
n_agents = env.n_agents
rng = np.random.default_rng(42)

aq = config['auction']
inv = config['investment']
year_data = []

for year in range(config['simulation']['n_years']):
    auction_actions = rng.uniform(
        [aq['price_min'], 0.0, 0.0, -1.0, -1.0, -1.0],
        [aq['price_max'], aq['quantity_max'], inv['max_invest_frac'], 1.0, 1.0, 1.0],
        size=(n_agents, 6)).astype(np.float32)
    obs2, _ = env.step_auction(auction_actions)

    secondary_actions = rng.uniform(
        [0.5, -aq['quantity_max']], [2.0, aq['quantity_max']], size=(n_agents, 2)
    ).astype(np.float32)
    obs1, rewards, terminated, _, info = env.step_secondary(secondary_actions)
    log = info['year_log']

    row = {'year': year, 'cap': log['cap'], 'price': log['clearing_price']}
    for i in range(n_agents):
        row[f'green_A{i+1}'] = log['green_fracs'][i]
        row[f'emissions_A{i+1}'] = log['emissions'][i]
        row[f'reward_A{i+1}'] = rewards[i]
    year_data.append(row)
    if terminated:
        break

df = pd.DataFrame(year_data)
print(df[['year', 'cap', 'price']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
for i in range(n_agents):
    ax.plot(df['year'], df[f'green_A{i+1}'] * 100, label=f'A{i+1}', marker='o', ms=4)
ax.set_xlabel('Year'); ax.set_ylabel('Green Fraction (%)')
ax.set_title('Green Fraction (Random Policy)'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for i in range(n_agents):
    ax.plot(df['year'], df[f'emissions_A{i+1}'], label=f'A{i+1}', marker='o', ms=4)
ax.set_xlabel('Year'); ax.set_ylabel('Emissions (Mt CO\u2082)')
ax.set_title('Emissions (Random Policy)'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(df['year'], df['cap'], label='Cap (Mt)', marker='s', color='red')
ax2 = ax.twinx()
ax2.plot(df['year'], df['price'], label='Price (\u20ac/t)', marker='^', color='blue')
ax.set_xlabel('Year'); ax.set_ylabel('Cap (Mt)', color='red')
ax2.set_ylabel('Price (\u20ac/t)', color='blue')
ax.set_title('Cap Trajectory & Carbon Price'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Train PPO Agents (Short Run)

In [ ]:
from scripts.train import train_one_seed

train_config = config.copy()
train_config['simulation'] = dict(config['simulation'])
train_config['simulation']['n_episodes'] = 1500
train_config['logging'] = dict(config['logging'])
train_config['logging']['log_interval'] = 50
train_config['logging']['save_interval'] = 100

train_one_seed(train_config, seed=42)

## 6. Quick Results Check

In [ ]:
ep_df = pd.read_csv('results/training_log_s42.csv')
yr_df = pd.read_csv('results/year_log_s42.csv')
n_agents = config['companies']['n_agents']
agent_colors = [plt.cm.tab10(i) for i in range(n_agents)]
ROLL = 50

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'reward_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], label=f'A{i+1}', lw=0.9)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title('Per-Agent Reward (rolling mean)'); ax.legend(fontsize=7, ncol=4); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(ep_df['episode'], ep_df['clearing_price_last'].rolling(ROLL, min_periods=1).mean(), color='darkblue')
ax.set_xlabel('Episode'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Clearing Price (last year, rolling mean)'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'green_frac_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], label=f'A{i+1}', lw=0.9)
ax.set_xlabel('Episode'); ax.set_ylabel('Green Fraction')
ax.set_title('Green Fraction (rolling mean)'); ax.legend(fontsize=7, ncol=4); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.bar(ep_df['episode'], ep_df['secondary_volume'].rolling(ROLL, min_periods=1).mean(),
       width=max(1, len(ep_df)//200), alpha=0.5, color='steelblue', label='Volume (Mt)')
ax2 = ax.twinx()
ax2.plot(ep_df['episode'], ep_df['secondary_match_rate'].rolling(ROLL, min_periods=1).mean(),
         color='coral', lw=1.2, label='Match Rate')
ax.set_xlabel('Episode'); ax.set_ylabel('Volume (Mt)'); ax2.set_ylabel('Match Rate')
ax.set_title('Secondary Market'); ax.grid(True, alpha=0.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.suptitle('Quick Results Check', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 7. Full Training Run

Run for more episodes to allow agents to develop meaningful strategies.
The analysis sections below are designed for this longer run.

In [ ]:
from scripts.train import train_one_seed

full_config = config.copy()
full_config['simulation'] = dict(config['simulation'])
full_config['simulation']['n_episodes'] = 40000
full_config['logging'] = dict(config['logging'])
full_config['logging']['log_interval'] = 100
full_config['logging']['results_dir'] = 'results/'

train_one_seed(full_config, seed=42)

---
# Post-Training Analysis

> **Run the data-loading cell below first**, then jump to either part.

| Part | Purpose |
|------|---------|
| **Part I — Diagnostic** | Is the *simulation itself* realistic and mechanically correct? |
| **Part II — Analytical** | What *strategies* did the agents learn? |

In [ ]:
# ── Load & Prepare Data (run this first) ─────────────────────────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

ep_df = pd.read_csv('results/training_log_s42.csv')
yr_df = pd.read_csv('results/year_log_s42.csv')
n_agents = config['companies']['n_agents']

sns.set_style('whitegrid')
agent_colors = [plt.cm.tab10(i) for i in range(n_agents)]
_base_archetypes = ['Coal-Heavy', 'Gas-Dominant', 'Mixed', 'Near-Green']
agent_labels = []
for arch in _base_archetypes:
    for k in range(n_agents // 4):
        agent_labels.append(f'A{len(agent_labels)+1} ({arch})')
while len(agent_labels) < n_agents:
    agent_labels.append(f'A{len(agent_labels)+1}')

LAST_N = min(200, len(ep_df) // 4)
last_episodes = ep_df['episode'].unique()[-LAST_N:]
recent_ep = ep_df[ep_df['episode'].isin(last_episodes)].copy()
recent_yr = yr_df[yr_df['episode'].isin(last_episodes)].copy()
ROLL = 50

n_years_per_ep = yr_df.groupby('episode')['year'].count().mode().iloc[0]

print(f'Loaded {len(ep_df)} episodes, {len(yr_df)} year-rows')
print(f'Converged window: last {LAST_N} episodes  |  {n_years_per_ep} years/episode')
print(f'Agents: {n_agents}  |  Archetypes: {agent_labels}')

---
# Part I — Diagnostic Analysis

> *Does the simulation produce economically plausible outcomes?*
>
> We check: price levels, cap-emissions alignment, MSR/TNAC dynamics,
> auction & secondary market mechanics, stochastic shocks (P5/P6),
> and training convergence.

### D1. Carbon Price & Cap Schedule

The clearing price should broadly reflect real EU ETS dynamics:
- Phase 4 prices in the €50–100 range
- Prices rising as the cap tightens under the LRF
- TNAC declining over time as MSR absorbs surplus

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Clearing price over training
ax = axes[0, 0]
ax.plot(ep_df['episode'], ep_df['clearing_price_last'].rolling(ROLL, min_periods=1).mean(),
        color='darkblue', lw=1.2)
ax.axhspan(50, 100, alpha=0.08, color='green', label='EU ETS Phase 4 range')
ax.set_xlabel('Episode'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Clearing Price Convergence'); ax.legend(); ax.grid(True, alpha=0.3)

# 2 — Price distribution (converged)
ax = axes[0, 1]
prices_conv = recent_yr.groupby('episode')['clearing_price'].last()
ax.hist(prices_conv, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(prices_conv.median(), color='red', ls='--', label=f'Median={prices_conv.median():.0f}\u20ac')
ax.set_xlabel('Clearing Price (\u20ac/t)'); ax.set_ylabel('Count')
ax.set_title('Price Distribution (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 3 — Cap schedule vs total emissions
ax = axes[0, 2]
yr_agg = recent_yr.groupby('year').agg(
    cap=('cap', 'mean'),
    total_emissions=pd.NamedAgg(
        column='emissions_A1',
        aggfunc=lambda x: sum(recent_yr.loc[x.index, f'emissions_A{i+1}'] for i in range(n_agents)).mean()
    )
).reset_index()
# Recompute properly
cap_by_yr = recent_yr.groupby('year')['cap'].mean()
emis_cols = [f'emissions_A{i+1}' for i in range(n_agents)]
total_emis_by_yr = recent_yr.groupby('year')[emis_cols].sum().mean(axis=0)
# Better: per year, sum across agents, then mean across episodes
emis_per_ep_yr = recent_yr.copy()
emis_per_ep_yr['total_emis'] = sum(emis_per_ep_yr[f'emissions_A{i+1}'] for i in range(n_agents))
emis_by_yr = emis_per_ep_yr.groupby('year')['total_emis'].mean()

ax.plot(cap_by_yr.index, cap_by_yr.values, 'r-s', ms=5, label='Cap (Mt)', lw=2)
ax.plot(emis_by_yr.index, emis_by_yr.values, 'b-o', ms=4, label='Total Emissions (Mt)', lw=1.5)
ax.fill_between(cap_by_yr.index, cap_by_yr.values, emis_by_yr.values,
                alpha=0.15, color='gray')
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO\u2082')
ax.set_title('Cap vs Emissions (converged mean)'); ax.legend(); ax.grid(True, alpha=0.3)

# 4 — Within-episode price trajectory
ax = axes[1, 0]
sample_eps = sorted(recent_yr['episode'].unique()[-20:])
for ep in sample_eps:
    ep_data = recent_yr[recent_yr['episode'] == ep]
    ax.plot(ep_data['year'], ep_data['clearing_price'], color='steelblue', alpha=0.15, lw=0.8)
mean_price_yr = recent_yr.groupby('year')['clearing_price'].mean()
ax.plot(mean_price_yr.index, mean_price_yr.values, 'darkblue', lw=2.5, label='Mean')
ax.set_xlabel('Year'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Within-Episode Price Trajectory'); ax.legend(); ax.grid(True, alpha=0.3)

# 5 — TNAC over years
ax = axes[1, 1]
tnac_yr = recent_yr.groupby('year')['tnac'].agg(['mean', 'std'])
ax.plot(tnac_yr.index, tnac_yr['mean'], 'darkorange', lw=2, marker='o', ms=4)
ax.fill_between(tnac_yr.index, tnac_yr['mean'] - tnac_yr['std'],
                tnac_yr['mean'] + tnac_yr['std'], alpha=0.2, color='orange')
ax.set_xlabel('Year'); ax.set_ylabel('TNAC (Mt)')
ax.set_title('Allowances in Circulation (mean\u00b11\u03c3)'); ax.grid(True, alpha=0.3)

# 6 — MSR reserve
ax = axes[1, 2]
msr_yr = recent_yr.groupby('year')['msr_reserve'].mean()
ax.bar(msr_yr.index, msr_yr.values, color='mediumpurple', alpha=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('MSR Reserve (Mt)')
ax.set_title('Market Stability Reserve (converged mean)'); ax.grid(True, alpha=0.3)

plt.suptitle('D1 \u2014 Carbon Price & Cap Schedule Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D2. Auction Mechanics

Sanity checks for the uniform-price auction:
- Allocations should sum to roughly the auctioned volume
- Bid prices should bracket the clearing price
- No persistent auction failures

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Total allocation vs auction volume
ax = axes[0, 0]
alloc_cols = [f'alloc_A{i+1}' for i in range(n_agents)]
alloc_total = recent_yr[alloc_cols].sum(axis=1)
ax.scatter(recent_yr['auction_volume'], alloc_total, alpha=0.05, s=8, color='steelblue')
lims = [0, max(recent_yr['auction_volume'].max(), alloc_total.max()) * 1.1]
ax.plot(lims, lims, 'r--', lw=1, label='Perfect match')
ax.set_xlabel('Auction Volume (Mt)'); ax.set_ylabel('Total Allocated (Mt)')
ax.set_title('Allocation vs Supply'); ax.legend(); ax.grid(True, alpha=0.3)

# 2 — Per-agent allocation distribution
ax = axes[0, 1]
alloc_data = [recent_yr[f'alloc_A{i+1}'].values for i in range(n_agents)]
bp = ax.boxplot(alloc_data, labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.set_ylabel('Allocation (Mt)'); ax.set_title('Allocation Distribution (converged)')
ax.grid(True, alpha=0.3)

# 3 — Bid price vs clearing price
ax = axes[0, 2]
for i in range(n_agents):
    bid_roll = ep_df[f'bid_price_A{i+1}'].rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], bid_roll, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
clr_roll = ep_df['clearing_price_last'].rolling(ROLL, min_periods=1).mean()
ax.plot(ep_df['episode'], clr_roll, 'k--', lw=2, label='Clearing')
ax.set_xlabel('Episode'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Bid Prices vs Clearing Price'); ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — Auction cost per agent (converged)
ax = axes[1, 0]
acost_means = [recent_yr.groupby('episode')[f'auction_cost_A{i+1}'].sum().mean() for i in range(n_agents)]
bars = ax.bar([f'A{i+1}' for i in range(n_agents)], acost_means,
              color=agent_colors)
ax.set_ylabel('Mean Auction Cost / Episode (M\u20ac)')
ax.set_title('Auction Cost by Agent (converged)'); ax.grid(True, alpha=0.3)

# 5 — Under-allocation frequency
ax = axes[1, 1]
# Check how often total allocation < 90% of auction volume
under = alloc_total < 0.9 * recent_yr['auction_volume']
under_rate_yr = under.groupby(recent_yr['year']).mean()
ax.bar(under_rate_yr.index, under_rate_yr.values * 100, color='salmon', alpha=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Under-allocation Rate (%)')
ax.set_title('Auction Under-Subscription Frequency'); ax.grid(True, alpha=0.3)

# 6 — Price volatility over training
ax = axes[1, 2]
ax.plot(ep_df['episode'], ep_df['price_std'].rolling(ROLL, min_periods=1).mean(),
        color='teal', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('Within-Episode Price Std (\u20ac)')
ax.set_title('Intra-Episode Price Volatility'); ax.grid(True, alpha=0.3)

plt.suptitle('D2 \u2014 Auction Mechanics Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D3. Secondary Market Mechanics

Does the double-auction secondary market function properly?
- Reasonable match rates (not all-or-nothing)
- Secondary prices tracking auction prices
- Both buyers and sellers participating

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Match rate over training
ax = axes[0, 0]
ax.plot(ep_df['episode'], ep_df['secondary_match_rate'].rolling(ROLL, min_periods=1).mean(),
        color='coral', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Match Rate')
ax.set_title('Secondary Market Match Rate'); ax.grid(True, alpha=0.3)

# 2 — Secondary volume over training
ax = axes[0, 1]
ax.plot(ep_df['episode'], ep_df['secondary_volume'].rolling(ROLL, min_periods=1).mean(),
        color='steelblue', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Volume (Mt)')
ax.set_title('Secondary Market Volume'); ax.grid(True, alpha=0.3)

# 3 — Secondary vs Auction price
ax = axes[0, 2]
sec_mean_yr = recent_yr.groupby('year')['secondary_price'].mean()
auc_mean_yr = recent_yr.groupby('year')['clearing_price'].mean()
ax.plot(auc_mean_yr.index, auc_mean_yr.values, 'b-o', ms=4, label='Auction', lw=1.5)
ax.plot(sec_mean_yr.index, sec_mean_yr.values, 'r-s', ms=4, label='Secondary', lw=1.5)
ax.set_xlabel('Year'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Auction vs Secondary Price (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 4 — Net trade by agent by year (who buys/sells)
ax = axes[1, 0]
for i in range(n_agents):
    trade_by_yr = recent_yr.groupby('year')[f'trade_qty_A{i+1}'].mean()
    ax.plot(trade_by_yr.index, trade_by_yr.values, color=agent_colors[i],
            marker='o', ms=3, label=f'A{i+1}', lw=1)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Net Trade Qty (Mt)')
ax.set_title('Mean Trade Position by Year (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 5 — Buy/sell balance: count of buyers vs sellers per year
ax = axes[1, 1]
buy_count_yr = []
sell_count_yr = []
for yr in range(n_years_per_ep):
    yr_data = recent_yr[recent_yr['year'] == yr]
    n_buy = sum((yr_data[f'trade_qty_A{i+1}'] > 0.01).mean() for i in range(n_agents))
    n_sell = sum((yr_data[f'trade_qty_A{i+1}'] < -0.01).mean() for i in range(n_agents))
    buy_count_yr.append(n_buy)
    sell_count_yr.append(n_sell)
x = np.arange(n_years_per_ep)
ax.bar(x - 0.15, buy_count_yr, 0.3, color='forestgreen', alpha=0.8, label='Buyers')
ax.bar(x + 0.15, sell_count_yr, 0.3, color='firebrick', alpha=0.8, label='Sellers')
ax.set_xlabel('Year'); ax.set_ylabel('Avg # Agents')
ax.set_title('Buyers vs Sellers per Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Secondary net cost distribution
ax = axes[1, 2]
snet_data = []
for i in range(n_agents):
    ep_snet = recent_yr.groupby('episode')[f'secondary_net_A{i+1}'].sum()
    snet_data.append(ep_snet.values)
bp = ax.boxplot(snet_data, labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.axhline(0, color='black', ls='--', lw=0.5)
ax.set_ylabel('Net Secondary Cost / Episode (M\u20ac)')
ax.set_title('Secondary Net Cost (+paid / -received)')
ax.grid(True, alpha=0.3)

plt.suptitle('D3 \u2014 Secondary Market Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D4. Stochastic Shocks (P5 & P6)

Verify that emission demand shocks (P5) and construction uncertainty (P6) are
active and producing the intended distributions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

shock_cols = [f'emission_shock_A{i+1}' for i in range(n_agents)]
cf_cols = [f'cf_shock_A{i+1}' for i in range(n_agents)]
cancel_cols = [f'cancellation_A{i+1}' for i in range(n_agents)]

has_shocks = any(c in recent_yr.columns for c in shock_cols) and recent_yr[shock_cols[0]].std() > 1e-6

# 1 — P5 shock distribution
ax = axes[0, 0]
if has_shocks:
    all_shocks = pd.concat([recent_yr[c] * 100 for c in shock_cols if c in recent_yr.columns])
    ax.hist(all_shocks, bins=60, color='steelblue', edgecolor='white', alpha=0.8, density=True)
    ax.set_xlabel('Emission Shock \u03b5 (%)')
    ax.set_title(f'P5 Shock Distribution (\u03c3={all_shocks.std():.1f}%)')
else:
    ax.text(0.5, 0.5, 'P5 shocks inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P5 \u2014 Inactive')
ax.grid(True, alpha=0.3)

# 2 — P5 cross-agent correlation
ax = axes[0, 1]
if has_shocks and n_agents >= 2:
    s1 = recent_yr[shock_cols[0]] * 100
    s2 = recent_yr[shock_cols[1]] * 100
    ax.scatter(s1, s2, alpha=0.05, s=5, color='steelblue')
    corr = s1.corr(s2)
    ax.set_xlabel(f'A1 shock (%)'); ax.set_ylabel(f'A2 shock (%)')
    ax.set_title(f'P5 Cross-Agent Correlation (\u03c1={corr:.2f})')
else:
    ax.text(0.5, 0.5, 'P5 inactive or <2 agents', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 3 — P6 CF noise
ax = axes[1, 0]
has_cf = any(c in recent_yr.columns for c in cf_cols) and recent_yr[cf_cols[0]].std() > 1e-8
if has_cf:
    all_cf = pd.concat([recent_yr[c] * 100 for c in cf_cols if c in recent_yr.columns])
    ax.hist(all_cf, bins=60, color='seagreen', edgecolor='white', alpha=0.8, density=True)
    ax.set_xlabel('CF Noise (%)')
    ax.set_title(f'P6 Capacity-Factor Noise (\u03c3={all_cf.std():.1f}%)')
else:
    ax.text(0.5, 0.5, 'P6 CF-noise inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P6 CF \u2014 Inactive')
ax.grid(True, alpha=0.3)

# 4 — P6 cancellations
ax = axes[1, 1]
has_cancel = any(c in recent_yr.columns for c in cancel_cols) and recent_yr[cancel_cols[0]].sum() > 0
if has_cancel:
    cancel_per_ep = recent_yr.groupby('episode')[cancel_cols].sum().sum(axis=1)
    ax.hist(cancel_per_ep, bins=30, color='coral', edgecolor='white', alpha=0.8)
    ax.set_xlabel('Total Cancellations / Episode')
    ax.set_title(f'P6 Project Cancellations (mean={cancel_per_ep.mean():.1f}/ep)')
else:
    ax.text(0.5, 0.5, 'P6 cancellations inactive', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('P6 Cancellations \u2014 Inactive')
ax.grid(True, alpha=0.3)

plt.suptitle('D4 \u2014 Stochastic Shock Diagnostics (P5 & P6)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# Summary
print(f'P5 emission shock active: {has_shocks}')
print(f'P6 CF-noise active:       {has_cf}')
print(f'P6 cancellation active:   {has_cancel}')

### D5. Compliance & Penalty Mechanics

Verify that compliance mechanics work:
- Shortfalls are detected and penalized
- Penalties are proportional to shortfall
- Banking (carry-over) functions correctly

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Shortfall rate over training
ax = axes[0, 0]
for i in range(n_agents):
    sf = (ep_df[f'shortfall_A{i+1}'] > 1e-6).astype(float).rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], sf * 100, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Episodes with Shortfall (%)')
ax.set_title('Shortfall Frequency'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 2 — Penalty vs shortfall (should be correlated)
ax = axes[0, 1]
for i in range(n_agents):
    ax.scatter(recent_yr[f'shortfall_A{i+1}'], recent_yr[f'penalty_A{i+1}'],
               alpha=0.05, s=5, color=agent_colors[i], label=f'A{i+1}')
ax.set_xlabel('Shortfall (Mt)'); ax.set_ylabel('Penalty (M\u20ac)')
ax.set_title('Penalty vs Shortfall (should be proportional)')
ax.grid(True, alpha=0.3)

# 3 — Holdings by year (banking)
ax = axes[0, 2]
for i in range(n_agents):
    hold_yr = recent_yr.groupby('year')[f'holdings_A{i+1}'].mean()
    ax.plot(hold_yr.index, hold_yr.values, color=agent_colors[i], marker='o', ms=3,
            label=f'A{i+1}', lw=1)
ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
ax.set_title('Banking: Holdings by Year (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Compliance surplus distribution
ax = axes[1, 0]
if f'compliance_surplus_A1' in recent_yr.columns:
    surp_data = [recent_yr.groupby('episode')[f'compliance_surplus_A{i+1}'].mean().values
                 for i in range(n_agents)]
    bp = ax.boxplot(surp_data, labels=[f'A{i+1}' for i in range(n_agents)],
                    patch_artist=True, showfliers=False)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(agent_colors[i])
    ax.axhline(0, color='red', ls='--', lw=0.5)
    ax.set_ylabel('Mean Compliance Surplus (Mt)')
    ax.set_title('Compliance Surplus (+over / -under)')
else:
    ax.text(0.5, 0.5, 'compliance_surplus not logged', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 5 — Total penalty over training
ax = axes[1, 1]
pen_cols = [f'penalty_A{i+1}' for i in range(n_agents)]
total_pen = ep_df[pen_cols].sum(axis=1)
ax.plot(ep_df['episode'], total_pen.rolling(ROLL, min_periods=1).mean(), color='firebrick', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Penalty (M\u20ac)')
ax.set_title('System-Wide Penalty Trend'); ax.grid(True, alpha=0.3)

# 6 — Carry-forward: bank_end vs bank_start
ax = axes[1, 2]
if 'bank_end_A1' in recent_yr.columns and 'bank_start_A1' in recent_yr.columns:
    for i in range(min(4, n_agents)):
        bs = recent_yr.groupby('year')[f'bank_start_A{i+1}'].mean()
        be = recent_yr.groupby('year')[f'bank_end_A{i+1}'].mean()
        ax.plot(bs.index, bs.values, color=agent_colors[i], ls='--', lw=1, alpha=0.7)
        ax.plot(be.index, be.values, color=agent_colors[i], ls='-', lw=1.5,
                label=f'A{i+1} (solid=end, dash=start)')
    ax.set_xlabel('Year'); ax.set_ylabel('Bank (Mt)')
    ax.set_title('Banking: Start vs End of Year (first 4 agents)')
    ax.legend(fontsize=6, ncol=2)
else:
    ax.text(0.5, 0.5, 'bank_end not logged', ha='center', va='center', transform=ax.transAxes)
ax.grid(True, alpha=0.3)

plt.suptitle('D5 \u2014 Compliance & Penalty Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### D6. Training Convergence

Check that neural networks are learning stably:
- Losses decreasing
- Entropy decay triggered at the right time
- Rewards stabilizing

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — Total reward
ax = axes[0, 0]
rew_cols = [f'reward_A{i+1}' for i in range(n_agents)]
total_rew = ep_df[rew_cols].sum(axis=1)
ax.plot(ep_df['episode'], total_rew.rolling(ROLL, min_periods=1).mean(), color='darkgreen', lw=1.2)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.set_title('System Reward Convergence'); ax.grid(True, alpha=0.3)

# 2 — Per-agent reward
ax = axes[0, 1]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'reward_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title('Per-Agent Reward'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Actor loss
ax = axes[0, 2]
for i in range(n_agents):
    col = f'actor_loss_A{i+1}'
    if col in ep_df.columns:
        vals = ep_df[col].replace(0, np.nan).rolling(ROLL, min_periods=1).mean()
        ax.plot(ep_df['episode'], vals, color=agent_colors[i], lw=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Actor Loss')
ax.set_title('Actor Loss'); ax.grid(True, alpha=0.3)

# 4 — Critic loss
ax = axes[1, 0]
for i in range(n_agents):
    col = f'critic_loss_A{i+1}'
    if col in ep_df.columns:
        vals = ep_df[col].replace(0, np.nan).rolling(ROLL, min_periods=1).mean()
        ax.plot(ep_df['episode'], vals, color=agent_colors[i], lw=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Critic Loss')
ax.set_title('Critic Loss'); ax.grid(True, alpha=0.3)

# 5 — Entropy coefficient
ax = axes[1, 1]
ax.plot(ep_df['episode'], ep_df['entropy_coef'], color='purple', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('Entropy Coef')
ax.set_title('Entropy Coefficient Decay'); ax.grid(True, alpha=0.3)

# 6 — Epsilon
ax = axes[1, 2]
ax.plot(ep_df['episode'], ep_df['epsilon'], color='teal', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('\u03b5')
ax.set_title('Exploration \u03b5 Decay'); ax.grid(True, alpha=0.3)

plt.suptitle('D6 \u2014 Training Convergence Diagnostics', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

---
# Part II — Analytical: Agent Strategy Analysis

> *What strategies did the agents learn and why?*
>
> We decompose agent behaviour across the five decision phases:
> auction bidding, secondary-market trading, green investment,
> compliance/banking, and overall reward structure.

### A1. Auction Bidding Strategy

How agents bid in the primary EUA auction:
- **Price discovery** — do bids converge to a common clearing price?
- **Quantity strategy** — do agents bid for what they need or over/under-bid?
- **Archetype differences** — do coal-heavy agents bid differently from near-green?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Bid price evolution per agent
ax = axes[0, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'bid_price_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=agent_labels[i])
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Bid Price (\u20ac/t)')
ax.set_title('Bid Price Evolution'); ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Bid price distribution (converged)
ax = axes[0, 1]
bid_data = [recent_ep[f'bid_price_A{i+1}'].values for i in range(n_agents)]
parts = ax.violinplot(bid_data, positions=range(n_agents), showmedians=True, showextrema=False)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(agent_colors[i]); pc.set_alpha(0.7)
ax.set_xticks(range(n_agents)); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Bid Price (\u20ac/t)')
ax.set_title('Bid Price Distribution (converged)'); ax.grid(True, alpha=0.3)

# 3 — Allocation vs Emissions scatter (converged)
ax = axes[0, 2]
for i in range(n_agents):
    allocs = recent_yr.groupby('episode')[f'alloc_A{i+1}'].mean()
    emiss = recent_yr.groupby('episode')[f'emissions_A{i+1}'].mean()
    ax.scatter(allocs, emiss, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5, label='Alloc=Emissions')
ax.set_xlabel('Mean Allocation (Mt)'); ax.set_ylabel('Mean Emissions (Mt)')
ax.set_title('Allocation vs Emissions Need')
ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — Bid quantity evolution
ax = axes[1, 0]
for i in range(n_agents):
    alloc_roll = recent_yr.groupby('episode')[f'alloc_A{i+1}'].mean()
    alloc_smooth = alloc_roll.rolling(ROLL // 2, min_periods=1).mean()
    ax.plot(alloc_smooth.index, alloc_smooth.values, color=agent_colors[i], lw=0.9, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Allocation (Mt)')
ax.set_title('Allocation Over Training (proxy for bid qty)'); ax.legend(fontsize=6, ncol=4)
ax.grid(True, alpha=0.3)

# 5 — Bid price by year within episode (converged)
ax = axes[1, 1]
for i in range(n_agents):
    bid_yr = recent_yr.groupby('year')[f'bid_price_A{i+1}'].mean()
    ax.plot(bid_yr.index, bid_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Bid Price (\u20ac/t)')
ax.set_title('Within-Episode Bid Price by Year')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 6 — Coverage ratio: allocation / emissions
ax = axes[1, 2]
cov_data = []
for i in range(n_agents):
    a = recent_yr[f'alloc_A{i+1}']
    e = recent_yr[f'emissions_A{i+1}'].replace(0, np.nan)
    cov_data.append((a / e).dropna().values)
bp = ax.boxplot(cov_data, labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Full coverage')
ax.set_ylabel('Allocation / Emissions')
ax.set_title('Coverage Ratio (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('A1 \u2014 Auction Bidding Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A2. Secondary Market Strategy

How agents trade on the secondary market:
- **Role evolution** — who becomes a net buyer vs seller over training?
- **Year-by-year patterns** — do roles shift within an episode as the cap tightens?
- **Price premiums** — secondary vs auction price spread
- **Cost/revenue** — who profits from secondary trading?

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# 1 — Net trade position over training (rolling)
ax = axes[0, 0]
for i in range(n_agents):
    net_trade = yr_df.groupby('episode')[f'trade_qty_A{i+1}'].sum()
    ax.plot(net_trade.index, net_trade.rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=agent_labels[i])
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Net Trade (Mt)')
ax.set_title('Net Secondary Position Over Training')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Buy vs Sell volume per agent (converged)
ax = axes[0, 1]
buy_vols = []
sell_vols = []
for i in range(n_agents):
    tq = recent_yr[f'trade_qty_A{i+1}']
    buy_vols.append(tq[tq > 0.01].sum() / LAST_N)
    sell_vols.append(abs(tq[tq < -0.01].sum()) / LAST_N)
x = np.arange(n_agents)
ax.bar(x - 0.15, buy_vols, 0.3, color='forestgreen', alpha=0.8, label='Bought')
ax.bar(x + 0.15, sell_vols, 0.3, color='firebrick', alpha=0.8, label='Sold')
ax.set_xticks(x); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Mean Volume / Episode (Mt)')
ax.set_title('Buy vs Sell Volume (converged)'); ax.legend(); ax.grid(True, alpha=0.3)

# 3 — Trade position by year within episode (heatmap)
ax = axes[1, 0]
trade_matrix = np.zeros((n_agents, n_years_per_ep))
for i in range(n_agents):
    trade_by_yr = recent_yr.groupby('year')[f'trade_qty_A{i+1}'].mean()
    trade_matrix[i, :len(trade_by_yr)] = trade_by_yr.values
vmax = max(abs(trade_matrix.min()), abs(trade_matrix.max()))
im = ax.imshow(trade_matrix, aspect='auto', cmap='RdYlGn', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Year'); ax.set_ylabel('Agent')
ax.set_yticks(range(n_agents)); ax.set_yticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_title('Mean Trade Qty by Agent \u00d7 Year (green=buy, red=sell)')
plt.colorbar(im, ax=ax, label='Trade Qty (Mt)')

# 4 — Net secondary cost over training
ax = axes[1, 1]
for i in range(n_agents):
    snet = yr_df.groupby('episode')[f'secondary_net_A{i+1}'].sum()
    ax.plot(snet.index, snet.rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=0.9, label=f'A{i+1}')
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Net Cost (M\u20ac)')
ax.set_title('Secondary Net Cost Over Training (+paid / -received)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 5 — Secondary price premium over auction
ax = axes[2, 0]
premium = recent_yr['secondary_price'] / recent_yr['clearing_price'].replace(0, np.nan)
premium_yr = premium.groupby(recent_yr['year']).agg(['mean', 'std'])
ax.plot(premium_yr.index, premium_yr['mean'], 'b-o', ms=4, lw=1.5)
ax.fill_between(premium_yr.index,
                premium_yr['mean'] - premium_yr['std'],
                premium_yr['mean'] + premium_yr['std'], alpha=0.2)
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Par (1.0)')
ax.set_xlabel('Year'); ax.set_ylabel('Secondary / Auction Price')
ax.set_title('Secondary Price Premium by Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Trade qty vs compliance surplus (who trades based on need?)
ax = axes[2, 1]
if 'compliance_surplus_A1' in recent_yr.columns:
    for i in range(n_agents):
        ax.scatter(recent_yr[f'compliance_surplus_A{i+1}'],
                   recent_yr[f'trade_qty_A{i+1}'],
                   alpha=0.04, s=8, color=agent_colors[i], label=f'A{i+1}')
    ax.axhline(0, color='black', ls='-', lw=0.5)
    ax.axvline(0, color='black', ls='-', lw=0.5)
    ax.set_xlabel('Compliance Surplus (Mt)')
    ax.set_ylabel('Trade Qty (Mt, +buy/-sell)')
    ax.set_title('Trading vs Compliance Need')
    ax.legend(fontsize=6, ncol=4)
else:
    ax.text(0.5, 0.5, 'compliance_surplus not logged', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

plt.suptitle('A2 \u2014 Secondary Market Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A3. Green Transition Strategy

Investment patterns in renewable energy:
- **Transition speed** — how quickly does each archetype green its portfolio?
- **Investment intensity** — how aggressively do agents invest?
- **Technology preference** — onshore wind, offshore wind, or solar?
- **Construction pipeline** — queuing effects and timing

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Green fraction evolution over training
ax = axes[0, 0]
for i in range(n_agents):
    ax.plot(ep_df['episode'], ep_df[f'green_frac_A{i+1}'].rolling(ROLL, min_periods=1).mean(),
            color=agent_colors[i], lw=1, label=agent_labels[i])
ax.set_xlabel('Episode'); ax.set_ylabel('Green Fraction')
ax.set_title('Green Transition Over Training')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

# 2 — Green fraction by year within episode (converged)
ax = axes[0, 1]
for i in range(n_agents):
    gf_yr = recent_yr.groupby('year')[f'green_frac_A{i+1}'].mean()
    ax.plot(gf_yr.index, gf_yr.values * 100, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Green Fraction (%)')
ax.set_title('Within-Episode Green Trajectory (converged)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Delta green by year (when do agents invest?)
ax = axes[0, 2]
for i in range(n_agents):
    dg_yr = recent_yr.groupby('year')[f'delta_green_A{i+1}'].mean() * 100
    ax.plot(dg_yr.index, dg_yr.values, color=agent_colors[i], marker='s', ms=3,
            lw=1, label=f'A{i+1}')
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('\u0394 Green (%)')
ax.set_title('Investment Timing: \u0394Green by Year')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Investment cost per agent (converged)
ax = axes[1, 0]
inv_means = [recent_yr.groupby('episode')[f'invest_cost_A{i+1}'].sum().mean()
             for i in range(n_agents)]
ax.bar([f'A{i+1}' for i in range(n_agents)], inv_means, color=agent_colors)
ax.set_ylabel('Mean Investment Cost / Episode (M\u20ac)')
ax.set_title('Total Investment Spend (converged)'); ax.grid(True, alpha=0.3)

# 5 — Construction queue over training
ax = axes[1, 1]
for i in range(n_agents):
    col = f'queue_size_A{i+1}'
    if col in ep_df.columns:
        ax.plot(ep_df['episode'], ep_df[col].rolling(ROLL, min_periods=1).mean(),
                color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Queue Size')
ax.set_title('Construction Queue Over Training')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 6 — Green fraction vs emissions (converged)
ax = axes[1, 2]
for i in range(n_agents):
    gf = recent_yr.groupby('episode')[f'green_frac_A{i+1}'].last()
    em = recent_yr.groupby('episode')[f'emissions_A{i+1}'].mean()
    ax.scatter(gf * 100, em, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
ax.set_xlabel('Final Green Fraction (%)'); ax.set_ylabel('Mean Emissions (Mt)')
ax.set_title('Green Fraction vs Emissions')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A3 \u2014 Green Transition Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A4. Compliance & Banking Strategy

How agents manage their allowance portfolio:
- **Banking** — accumulating surplus for future compliance
- **Shortfall avoidance** — learning to stay compliant
- **Risk management** — holdings buffer vs cost minimization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Holdings trajectory by year (converged)
ax = axes[0, 0]
for i in range(n_agents):
    h_yr = recent_yr.groupby('year')[f'holdings_A{i+1}'].mean()
    ax.plot(h_yr.index, h_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
ax.set_title('Banking: Year-by-Year Holdings')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 2 — Shortfall per agent over training
ax = axes[0, 1]
for i in range(n_agents):
    sf = ep_df[f'shortfall_A{i+1}'].rolling(ROLL, min_periods=1).mean()
    ax.plot(ep_df['episode'], sf, color=agent_colors[i], lw=0.8, label=f'A{i+1}')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Shortfall (Mt)')
ax.set_title('Shortfall Reduction Over Training')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Holdings vs emissions need (buffer ratio)
ax = axes[0, 2]
for i in range(n_agents):
    h = recent_yr[f'holdings_A{i+1}']
    e = recent_yr[f'emissions_A{i+1}'].replace(0, np.nan)
    buffer = (h / e).dropna()
    buf_yr = buffer.groupby(recent_yr.loc[buffer.index, 'year']).mean()
    ax.plot(buf_yr.index, buf_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1, label=f'A{i+1}')
ax.axhline(1.0, color='red', ls='--', lw=0.8, label='Need=1.0')
ax.set_xlabel('Year'); ax.set_ylabel('Holdings / Emissions')
ax.set_title('Bank Buffer Ratio by Year')
ax.legend(fontsize=6, ncol=3); ax.grid(True, alpha=0.3)

# 4 — MAC fuel-switching usage
ax = axes[1, 0]
mac_cols_ep = [f'total_mac_reduction_A{i+1}' for i in range(n_agents)
               if f'total_mac_reduction_A{i+1}' in ep_df.columns]
if mac_cols_ep:
    for i, col in enumerate(mac_cols_ep):
        ax.plot(ep_df['episode'], ep_df[col].rolling(ROLL, min_periods=1).mean(),
                color=agent_colors[i], lw=0.8, label=f'A{i+1}')
    ax.set_xlabel('Episode'); ax.set_ylabel('MAC Reduction (Mt)')
    ax.set_title('Fuel Switching (Coal\u2192Gas)')
    ax.legend(fontsize=6, ncol=4)
else:
    ax.text(0.5, 0.5, 'MAC data not available', ha='center', va='center',
            transform=ax.transAxes)
ax.grid(True, alpha=0.3)

# 5 — Penalty by archetype
ax = axes[1, 1]
pen_by_arch = {}
for i in range(n_agents):
    arch = agent_labels[i].split('(')[1].rstrip(')') if '(' in agent_labels[i] else f'A{i+1}'
    pen = recent_ep[f'penalty_A{i+1}'].mean()
    pen_by_arch.setdefault(arch, []).append(pen)
arch_names = list(pen_by_arch.keys())
arch_means = [np.mean(v) for v in pen_by_arch.values()]
ax.bar(arch_names, arch_means, color=['#555555', '#FF8C00', '#2E8B57', '#1E90FF'][:len(arch_names)])
ax.set_ylabel('Mean Penalty / Episode (M\u20ac)')
ax.set_title('Penalty by Archetype (converged)')
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
ax.grid(True, alpha=0.3)

# 6 — Emissions trajectory by year (converged)
ax = axes[1, 2]
for i in range(n_agents):
    em_yr = recent_yr.groupby('year')[f'emissions_A{i+1}'].mean()
    ax.plot(em_yr.index, em_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Emissions (Mt)')
ax.set_title('Emission Trajectories Within Episode')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A4 \u2014 Compliance & Banking Strategy', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A5. Reward Decomposition & Cost Structure

What drives agent rewards? Decompose into cost components to understand
the economic trade-offs agents face.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Reward distribution (converged)
ax = axes[0, 0]
rew_data = [recent_ep[f'reward_A{i+1}'].values for i in range(n_agents)]
bp = ax.boxplot(rew_data, labels=[f'A{i+1}' for i in range(n_agents)],
                patch_artist=True, showfliers=False)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(agent_colors[i])
ax.set_ylabel('Total Reward / Episode')
ax.set_title('Reward Distribution (converged)'); ax.grid(True, alpha=0.3)

# 2 — Cost breakdown per agent (converged means)
ax = axes[0, 1]
cost_components = {}
for comp, col_tmpl in [('Auction', 'auction_cost_A{}'),
                        ('SecondaryNet', 'secondary_net_A{}'),
                        ('Investment', 'invest_cost_A{}'),
                        ('Penalty', 'penalty_A{}')]:
    means = []
    for i in range(n_agents):
        col = col_tmpl.format(i+1)
        if col in recent_yr.columns:
            means.append(recent_yr.groupby('episode')[col].sum().mean())
        else:
            means.append(0)
    cost_components[comp] = means

x = np.arange(n_agents)
bottom = np.zeros(n_agents)
comp_colors = {'Auction': 'steelblue', 'SecondaryNet': 'coral',
               'Investment': 'seagreen', 'Penalty': 'firebrick'}
for comp, vals in cost_components.items():
    vals_pos = np.maximum(vals, 0)
    ax.bar(x, vals_pos, 0.6, bottom=bottom, label=comp, color=comp_colors[comp], alpha=0.85)
    bottom += vals_pos
ax.set_xticks(x); ax.set_xticklabels([f'A{i+1}' for i in range(n_agents)])
ax.set_ylabel('Cost (M\u20ac / episode)')
ax.set_title('Cost Breakdown by Agent'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# 3 — Reward vs green fraction
ax = axes[0, 2]
for i in range(n_agents):
    gf = recent_ep[f'green_frac_A{i+1}']
    rw = recent_ep[f'reward_A{i+1}']
    ax.scatter(gf * 100, rw, alpha=0.15, s=12, color=agent_colors[i], label=f'A{i+1}')
ax.set_xlabel('Final Green Fraction (%)'); ax.set_ylabel('Total Reward')
ax.set_title('Reward vs Green Fraction')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Reward by archetype over training
ax = axes[1, 0]
n_per_arch = n_agents // 4
for a_idx, arch in enumerate(_base_archetypes):
    agent_ids = list(range(a_idx * n_per_arch, (a_idx + 1) * n_per_arch))
    if not agent_ids:
        continue
    arch_rew = sum(ep_df[f'reward_A{i+1}'] for i in agent_ids) / len(agent_ids)
    ax.plot(ep_df['episode'], arch_rew.rolling(ROLL, min_periods=1).mean(),
            lw=1.5, label=arch)
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Reward')
ax.set_title('Reward by Archetype'); ax.legend(); ax.grid(True, alpha=0.3)

# 5 — Cost by year within episode
ax = axes[1, 1]
if 'auction_cost_A1' in recent_yr.columns:
    for comp, col_tmpl, color in [('Auction', 'auction_cost_A{}', 'steelblue'),
                                   ('Investment', 'invest_cost_A{}', 'seagreen')]:
        total = sum(recent_yr[col_tmpl.format(i+1)] for i in range(n_agents))
        by_yr = total.groupby(recent_yr['year']).mean()
        ax.plot(by_yr.index, by_yr.values, marker='o', ms=4, lw=1.5, color=color, label=comp)
    ax.set_xlabel('Year'); ax.set_ylabel('Total Cost (M\u20ac)')
    ax.set_title('System Cost by Year'); ax.legend(); ax.grid(True, alpha=0.3)

# 6 — Reward per year within episode
ax = axes[1, 2]
for i in range(n_agents):
    rew_yr = recent_yr.groupby('year')[f'reward_A{i+1}'].mean()
    ax.plot(rew_yr.index, rew_yr.values, color=agent_colors[i], marker='o', ms=3,
            lw=1, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Reward')
ax.set_title('Reward by Year Within Episode')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A5 \u2014 Reward Decomposition & Cost Structure', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### A6. Deterministic Evaluation of Best Agents

Load the best checkpoint and run a single deterministic episode to see
learned strategies without exploration noise.

In [ ]:
from scripts.train import build_agents

eval_env = ETSEnvironment(config, seed=0)
eval_agents = build_agents(eval_env, config, seed=0)
n_agents = config['companies']['n_agents']
tech_names = config['technologies']['names']

ckpt_dir = 'results/checkpoints_s42'
for i, agent in enumerate(eval_agents):
    ckpt_path = os.path.join(ckpt_dir, f'agent_{i}_best.pt')
    if os.path.exists(ckpt_path):
        agent.load(ckpt_path)
        print(f'Loaded agent {i}')
    else:
        print(f'No checkpoint for agent {i}')

obs1, _ = eval_env.reset(seed=0)
eval_data = []

for year in range(config['simulation']['n_years']):
    auction_actions = np.zeros((n_agents, 6), dtype=np.float32)
    for i in range(n_agents):
        action, _, _ = eval_agents[i].select_auction_action(obs1[i], deterministic=True)
        auction_actions[i] = action
    obs2, _ = eval_env.step_auction(auction_actions)

    secondary_actions = np.zeros((n_agents, 2), dtype=np.float32)
    for i in range(n_agents):
        action, _, _ = eval_agents[i].select_secondary_action(obs2[i], deterministic=True)
        secondary_actions[i] = action

    obs1, rewards, terminated, _, info = eval_env.step_secondary(secondary_actions)
    log = info['year_log']

    row = {'year': year, 'cap': log['cap'], 'price': log['clearing_price'],
           'secondary_price': log.get('secondary_clearing', 0), 'tnac': log.get('tnac', 0)}
    for i in range(n_agents):
        row[f'green_A{i+1}'] = log['green_fracs'][i]
        row[f'emissions_A{i+1}'] = log['emissions'][i]
        row[f'alloc_A{i+1}'] = log['allocations'][i]
        row[f'holdings_A{i+1}'] = log['holdings'][i]
        row[f'trade_qty_A{i+1}'] = log['trade_qtys'][i]
        row[f'trade_cost_A{i+1}'] = log['trade_costs'][i]
        row[f'shortfall_A{i+1}'] = log['shortfalls'][i]
        row[f'penalty_A{i+1}'] = log['penalties'][i]
        row[f'reward_A{i+1}'] = log['rewards'][i]
        row[f'invest_cost_A{i+1}'] = log['invest_costs'][i]
        for t in range(5):
            row[f'mix_A{i+1}_{tech_names[t]}'] = log['tech_mixes'][i][t]
    eval_data.append(row)
    if terminated:
        break

eval_df = pd.DataFrame(eval_data)
print(f'Evaluation: {len(eval_df)} years')
print(eval_df[['year', 'cap', 'price', 'tnac']].to_string(index=False))

In [ ]:
# ── A6 Deterministic Evaluation Plots ─────────────────────────────────
tech_colors = {'coal': '#555555', 'gas': '#FF8C00', 'onshore_wind': '#2E8B57',
               'offshore_wind': '#1E90FF', 'solar': '#FFD700'}

# --- Tech mix: one subplot per agent ---
ncols = min(4, n_agents)
nrows_tech = (n_agents + ncols - 1) // ncols
fig_tech, axes_tech = plt.subplots(nrows_tech, ncols, figsize=(5*ncols, 4*nrows_tech), sharey=True)
axes_tech = np.atleast_1d(axes_tech).flatten()

for i in range(n_agents):
    ax = axes_tech[i]
    bottom_v = np.zeros(len(eval_df))
    for t in range(5):
        vals = eval_df[f'mix_A{i+1}_{tech_names[t]}'].values * 100
        ax.bar(eval_df['year'], vals, bottom=bottom_v,
               color=tech_colors[tech_names[t]], alpha=0.85,
               label=tech_names[t].replace('_', ' ').title())
        bottom_v += vals
    ax.set_title(f'{agent_labels[i]}', fontsize=10)
    ax.set_ylim(0, 100)
    if i % ncols == 0:
        ax.set_ylabel('Mix (%)')
    ax.set_xlabel('Year')

for j in range(n_agents, len(axes_tech)):
    axes_tech[j].set_visible(False)

handles, labels = axes_tech[0].get_legend_handles_labels()
fig_tech.legend(handles, labels, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig_tech.suptitle('A6a — Technology Mix Evolution (Deterministic)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# --- Key strategy plots ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 — Prices
ax = axes[0, 0]
ax.plot(eval_df['year'], eval_df['price'], 'b-o', ms=4, label='Auction', lw=1.5)
ax.plot(eval_df['year'], eval_df['secondary_price'], 'r-s', ms=4, label='Secondary', lw=1.5)
ax.set_xlabel('Year'); ax.set_ylabel('Price (\u20ac/t)')
ax.set_title('Price Trajectories'); ax.legend(); ax.grid(True, alpha=0.3)

# 2 — Trade patterns
ax = axes[0, 1]
for i in range(n_agents):
    ax.plot(eval_df['year'], eval_df[f'trade_qty_A{i+1}'], color=agent_colors[i],
            marker='o', ms=3, lw=1, label=f'A{i+1}')
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Trade Qty (Mt)')
ax.set_title('Secondary Trade Patterns')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 3 — Holdings
ax = axes[0, 2]
for i in range(n_agents):
    ax.plot(eval_df['year'], eval_df[f'holdings_A{i+1}'], color=agent_colors[i],
            marker='o', ms=3, lw=1, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Holdings (Mt)')
ax.set_title('Banking: Allowance Holdings')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 4 — Emissions vs Allocations
ax = axes[1, 0]
for i in range(n_agents):
    ax.plot(eval_df['year'], eval_df[f'emissions_A{i+1}'], color=agent_colors[i],
            ls='-', lw=1.2, label=f'A{i+1} emis')
    ax.plot(eval_df['year'], eval_df[f'alloc_A{i+1}'], color=agent_colors[i],
            ls='--', lw=0.8, alpha=0.6)
ax.set_xlabel('Year'); ax.set_ylabel('Mt')
ax.set_title('Emissions (solid) vs Allocation (dashed)')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 5 — Green fraction
ax = axes[1, 1]
for i in range(n_agents):
    ax.plot(eval_df['year'], eval_df[f'green_A{i+1}'] * 100, color=agent_colors[i],
            marker='o', ms=3, lw=1.2, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Green Fraction (%)')
ax.set_title('Green Transition'); ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

# 6 — Cumulative reward
ax = axes[1, 2]
for i in range(n_agents):
    cum_rew = eval_df[f'reward_A{i+1}'].cumsum()
    ax.plot(eval_df['year'], cum_rew, color=agent_colors[i], lw=1.5, label=f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Cumulative Reward')
ax.set_title('Cumulative Reward')
ax.legend(fontsize=6, ncol=4); ax.grid(True, alpha=0.3)

plt.suptitle('A6b — Deterministic Evaluation', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### Summary Statistics

Key metrics for thesis reporting.

In [ ]:
print('=' * 90)
print('TRAINING SUMMARY (converged: last {} episodes)'.format(LAST_N))
print('=' * 90)

# Per-agent summary
rows = []
for i in range(n_agents):
    row = {
        'Agent': agent_labels[i],
        'Reward': f"{recent_ep[f'reward_A{i+1}'].mean():.1f} \u00b1 {recent_ep[f'reward_A{i+1}'].std():.1f}",
        'Green%': f"{recent_ep[f'green_frac_A{i+1}'].mean()*100:.1f}%",
        'Penalty': f"{recent_ep[f'penalty_A{i+1}'].mean():.2f}",
        'Shortfall': f"{recent_ep[f'shortfall_A{i+1}'].mean():.3f}",
        'BidPrice': f"{recent_ep[f'bid_price_A{i+1}'].mean():.0f}\u20ac",
    }
    # Secondary market role
    net_trade = recent_yr[recent_yr['episode'].isin(last_episodes)].groupby('episode')[f'trade_qty_A{i+1}'].sum()
    mean_net = net_trade.mean()
    row['SecRole'] = 'Buyer' if mean_net > 0.1 else 'Seller' if mean_net < -0.1 else 'Balanced'
    row['SecNetMt'] = f"{mean_net:+.2f}"
    rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

print()
print(f"Mean clearing price (last year): {recent_ep['clearing_price_last'].mean():.1f} \u00b1 {recent_ep['clearing_price_last'].std():.1f} \u20ac/t")
print(f"Secondary match rate:            {recent_ep['secondary_match_rate'].mean()*100:.0f}%")
print(f"Secondary volume:                {recent_ep['secondary_volume'].mean():.1f} Mt/ep")
print(f"Mean system reward:              {recent_ep[[f'reward_A{i+1}' for i in range(n_agents)]].sum(axis=1).mean():.1f}")